# Session 07 — CNNs: Review & Modern Architectures

**Goal:** build classifiers of progressively more complex design — MLP → LeNet → VGG → ResNet → ConvNeXt — train each on Fruits-360 (10 grouped super-classes), and analyse what they learn.

This notebook tracks the slides one-for-one: every section corresponds to a building-block we covered in lecture.

> **Runtime → Change runtime type → GPU** (T4 is fine). On CPU the bigger blocks become slow.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/schedldave/cvi4ic-notebooks/blob/main/07-cnns.ipynb)

**Estimated runtime on a free Colab T4 GPU:** ~12 min for all 5 trainings at `EPOCHS=5`.
On CPU expect 30–45 min — switch to GPU before running.


## 0. Setup


In [ ]:
!pip install -q opencv-python-headless thop tqdm scikit-learn


In [ ]:
import os, math, random, time, json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Torch: {torch.__version__}  ·  Device: {device}")


## 1. Dataset — Fruits-360, grouped to 10 super-classes

We re-map the fine-grained Fruits-360 folders into 10 visually-distinct super-classes (Apple, Banana, Cherry, Cucumber, Grape, Peach, Pear, Pepper, Plum, Tomato).


In [ ]:
!git clone --depth 1 https://github.com/fruits-360/fruits-360-100x100.git 2>/dev/null || echo "(already cloned)"
DATA_ROOT = Path("fruits-360-100x100")
TRAIN_DIR = DATA_ROOT / "Training"
TEST_DIR  = DATA_ROOT / "Test"
print("train classes:", len(list(TRAIN_DIR.glob("*"))))


In [ ]:
CLASS_GROUPS = {
    "Apple":    ["Apple Braeburn 1", "Apple Golden 1", "Apple Granny Smith 1",
                 "Apple Pink Lady 1", "Apple Red Delicious 1"],
    "Banana":   ["Banana 1", "Banana 3", "Banana 4",
                 "Banana Lady Finger 1", "Banana Red 1"],
    "Cherry":   ["Cherry 1", "Cherry Rainier 1", "Cherry Sour 1",
                 "Cherry Wax Black 1", "Cherry Wax Yellow 1"],
    "Cucumber": ["Cucumber 1", "Cucumber 3", "Cucumber 5", "Cucumber 7", "Cucumber 9"],
    "Grape":    ["Grape 1", "Grape Blue 1", "Grape Pink 1", "Grape White 1", "Grape White 2"],
    "Peach":    ["Peach 1", "Peach 2", "Peach 3", "Peach 5", "Peach Flat 1"],
    "Pear":     ["Pear Abate 1", "Pear Forelle 1", "Pear Kaiser 1",
                 "Pear Red 1", "Pear Williams 1"],
    "Pepper":   ["Pepper 1", "Pepper Green 1", "Pepper Orange 1",
                 "Pepper Red 1", "Pepper Yellow 1"],
    "Plum":     ["Plum 1", "Plum 2", "Plum 3", "Plum 4", "Plum 5"],
    "Tomato":   ["Tomato 1", "Tomato Cherry Red 1", "Tomato Heart 1",
                 "Tomato Maroon 1", "Tomato Yellow 1"],
}
CLASS_NAMES = sorted(CLASS_GROUPS.keys())
NUM_CLASSES = len(CLASS_NAMES)
IMG_SIZE = 100

NORMALIZE = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
print(f"{NUM_CLASSES} super-classes:", CLASS_NAMES)


In [ ]:
class GroupedFruits(Dataset):
    """Fruits-360 remapped to 10 super-classes."""
    FOLDER_TO_GROUP = {folder: i for i, name in enumerate(CLASS_NAMES)
                       for folder in CLASS_GROUPS[name]}

    def __init__(self, root: Path, transform, samples_per_class: int | None = None):
        self.transform = transform
        self.samples = []
        for folder, group_idx in self.FOLDER_TO_GROUP.items():
            cls_dir = root / folder
            if not cls_dir.is_dir(): continue
            files = sorted(cls_dir.glob("*.jpg"))
            if samples_per_class is not None:
                share = max(1, samples_per_class // 5)
                files = files[:share]
            self.samples.extend((f, group_idx) for f in files)
        random.Random(SEED).shuffle(self.samples)

    def __len__(self):  return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        return self.transform(Image.open(path).convert("RGB")), label

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    NORMALIZE,
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    NORMALIZE,
])

train_ds = GroupedFruits(TRAIN_DIR, train_tf, samples_per_class=750)
test_ds  = GroupedFruits(TEST_DIR,  eval_tf,  samples_per_class=250)

BATCH = 128
# num_workers=0 keeps the notebook portable across Colab / Jupyter / Windows; 
# pin_memory only helps when a CUDA device is present.
pin = (device.type == "cuda")
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=pin)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=pin)
print(f"train: {len(train_ds)}  test: {len(test_ds)}")


In [ ]:
def denorm(img_t):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return (img_t.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

# One example per super-class
seen, examples = set(), []
for img, label in test_ds:
    if label not in seen:
        seen.add(label); examples.append((img, label))
    if len(seen) == NUM_CLASSES: break

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, (img, lbl) in zip(axes.flat, examples):
    ax.imshow(denorm(img)); ax.set_title(CLASS_NAMES[lbl]); ax.axis("off")
plt.suptitle("Fruits-360 — 10 super-classes (test subset)")
plt.tight_layout(); plt.show()


## 2. Training utilities

Same training loop for every model — only the architecture changes between sections.


In [ ]:
EPOCHS = 5      # bump to 10–15 if you have more time


def train_model(model, loaders, epochs=EPOCHS, lr=1e-3, weight_decay=1e-4, label="model"):
    train_loader, test_loader = loaders
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit  = nn.CrossEntropyLoss()
    hist  = {"train_acc": [], "test_acc": [], "train_loss": [], "test_loss": []}

    for ep in range(1, epochs + 1):
        model.train(); rl = rc = rt = 0
        for x, y in tqdm(train_loader, desc=f"[{label}] ep {ep}/{epochs}", leave=False):
            x, y = x.to(device), y.to(device)
            logits = model(x); loss = crit(logits, y)
            opt.zero_grad(); loss.backward(); opt.step()
            rl += loss.item() * x.size(0)
            rc += (logits.argmax(1) == y).sum().item(); rt += x.size(0)
        sched.step()
        train_loss, train_acc = rl/rt, rc/rt

        model.eval(); tl = tc = tt = 0
        with torch.no_grad():
            for x, y in test_loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                tl += crit(logits, y).item() * x.size(0)
                tc += (logits.argmax(1) == y).sum().item(); tt += x.size(0)
        test_loss, test_acc = tl/tt, tc/tt

        hist["train_loss"].append(train_loss); hist["train_acc"].append(train_acc)
        hist["test_loss"].append(test_loss);   hist["test_acc"].append(test_acc)
        print(f"  ep {ep:2d}/{epochs}  train={train_acc:.3f}  test={test_acc:.3f}  loss={train_loss:.3f}/{test_loss:.3f}")
    return hist


def param_count(m):
    return sum(p.numel() for p in m.parameters())


def model_flops(m, img_size=IMG_SIZE):
    """Forward-pass MAC count via thop. Returns FLOPs (MACs · 2)."""
    from thop import profile
    m.eval()
    macs, _ = profile(m, inputs=(torch.randn(1, 3, img_size, img_size).to(next(m.parameters()).device),),
                      verbose=False)
    return macs * 2  # FLOPs ≈ 2 × MACs


def plot_history(hist, label):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
    ep = range(1, len(hist["train_acc"]) + 1)
    ax[0].plot(ep, hist["train_loss"], label="train"); ax[0].plot(ep, hist["test_loss"], label="test")
    ax[0].set_title(f"{label} — loss");     ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(ep, hist["train_acc"],  label="train"); ax[1].plot(ep, hist["test_acc"],  label="test")
    ax[1].set_title(f"{label} — accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].set_ylim(0, 1)
    plt.tight_layout(); plt.show()


## 3. Baseline — MLP (no spatial structure)

Plain fully-connected network. Flattens the 100×100×3 image into a 30 000-d vector — already 15 M parameters in the first layer. **No locality, no weight sharing.**


In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim=3*IMG_SIZE*IMG_SIZE, hidden=(512, 256), num_classes=NUM_CLASSES):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(inplace=True), nn.Dropout(0.3)]
            prev = h
        layers += [nn.Linear(prev, num_classes)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x.flatten(1))

mlp = MLP().to(device)
print(f"MLP — params: {param_count(mlp):,}")


In [ ]:
hist_mlp = train_model(mlp, (train_loader, test_loader), label="MLP")
plot_history(hist_mlp, "MLP")


In [ ]:
# What did the MLP actually learn? Reshape the first 10 hidden weights back to image space.
W = mlp.net[0].weight.detach().cpu().numpy()  # (hidden_dim, 30000)
fig, axes = plt.subplots(1, 10, figsize=(16, 2))
for i, ax in enumerate(axes):
    t = W[i].reshape(3, IMG_SIZE, IMG_SIZE).transpose(1, 2, 0)
    t = (t - t.min()) / (t.max() - t.min() + 1e-9)
    ax.imshow(t); ax.axis("off"); ax.set_title(f"h{i}", fontsize=9)
plt.suptitle("MLP first-layer weights reshaped to image space — every hidden unit is a whole-image template")
plt.tight_layout(); plt.show()


## 4. LeNet-5 (1998) — the first practical CNN

Two convolutional stages (5×5 conv + ReLU + max-pool), then two fully-connected layers. Designed for handwritten-digit recognition; we apply it to colour 100×100 fruit photos by giving it 3 input channels and a slightly bigger first conv.

**What makes it different from the MLP:** locality (5×5 receptive field), weight sharing across positions, and a hierarchical feature representation.


In [ ]:
class LeNet(nn.Module):
    """LeNet-5, adapted to 100×100 RGB inputs.

    The slide shows the classic LeNet shape for 28×28 grayscale digits. At our
    100×100 RGB resolution we keep the same Conv5×5 + ReLU + MaxPool template
    but add one extra pooling stage so the fully-connected head stays small.
    """
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,  20, kernel_size=5, padding=2), nn.ReLU(inplace=True),  # 100×100
            nn.MaxPool2d(2),                                                      #  50×50
            nn.Conv2d(20, 50, kernel_size=5, padding=2), nn.ReLU(inplace=True),  #  50×50
            nn.MaxPool2d(2),                                                      #  25×25
            nn.Conv2d(50, 64, kernel_size=5, padding=2), nn.ReLU(inplace=True),  #  25×25
            nn.MaxPool2d(2),                                                      #  12×12
        )
        self.head = nn.Sequential(
            nn.Flatten(),                                                         # 64·12·12 = 9216
            nn.Linear(64 * 12 * 12, 120), nn.ReLU(inplace=True),
            nn.Linear(120, 84),           nn.ReLU(inplace=True),
            nn.Linear(84, num_classes),
        )
    def forward(self, x):
        return self.head(self.features(x))

lenet = LeNet().to(device)
print(lenet)
print(f"\nLeNet — params: {param_count(lenet):,}")


In [ ]:
hist_lenet = train_model(lenet, (train_loader, test_loader), label="LeNet")
plot_history(hist_lenet, "LeNet")


In [ ]:
# What did LeNet's first layer learn? With 3 input channels and 20 filters, each filter
# is a (3, 5, 5) tensor — visualise as a small RGB image.
W = lenet.features[0].weight.detach().cpu()  # (20, 3, 5, 5)
fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for i, ax in enumerate(axes.flat):
    t = W[i].permute(1, 2, 0).numpy()
    t = (t - t.min()) / (t.max() - t.min() + 1e-9)
    ax.imshow(t); ax.axis("off")
plt.suptitle("LeNet first-layer 5×5 RGB filters — already responding to colour & oriented edges")
plt.tight_layout(); plt.show()


## 5. VGG-style — stacks of 3×3 convs + BatchNorm

VGG (Simonyan & Zisserman, 2014) showed that **two stacked 3×3 convs replace a single 5×5** with fewer parameters and an extra non-linearity in between. Modern variants add BatchNorm between conv and activation.

Our `VGGTiny` stacks three (Conv 3×3 + BN + ReLU) ×2 + MaxPool blocks.


In [ ]:
class VGGBlock(nn.Module):
    """Two stacked 3×3 convs with BatchNorm + ReLU, then 2×2 max-pool."""
    def __init__(self, in_c, out_c):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
    def forward(self, x): return self.net(x)

class VGGTiny(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            VGGBlock(3, 32),    # 100 → 50
            VGGBlock(32, 64),   #  50 → 25
            VGGBlock(64, 128),  #  25 → 12
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Linear(128, num_classes)
    def forward(self, x):
        x = self.features(x)
        return self.fc(self.pool(x).flatten(1))

vgg = VGGTiny().to(device)
print(f"VGGTiny — params: {param_count(vgg):,}")


In [ ]:
hist_vgg = train_model(vgg, (train_loader, test_loader), label="VGGTiny")
plot_history(hist_vgg, "VGGTiny")


## 6. ResNet — residual blocks

He et al. 2016 introduced the **residual connection** `y = x + f(x)`. The network now learns a *small update* on top of the identity instead of the whole mapping, which dramatically eases optimisation of deep networks.

Our `TinyResNet` stacks three stages, each with two `BasicBlock`s. The first block of each stage downsamples (stride 2) and uses a 1×1 conv on the shortcut to match channels.


In [ ]:
class BasicBlock(nn.Module):
    """Two 3×3 convs with BN, residual connection. Downsamples when stride=2."""
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_c)
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_c, out_c, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_c),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)
        return F.relu(out, inplace=True)


class TinyResNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
        )
        # Three stages; each first block does stride-2 downsampling.
        self.stage1 = nn.Sequential(BasicBlock(32, 32),               BasicBlock(32, 32))    # 100
        self.stage2 = nn.Sequential(BasicBlock(32, 64,  stride=2),    BasicBlock(64, 64))    #  50
        self.stage3 = nn.Sequential(BasicBlock(64, 128, stride=2),    BasicBlock(128, 128))  #  25
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x); x = self.stage2(x); x = self.stage3(x)
        return self.fc(self.pool(x).flatten(1))

resnet = TinyResNet().to(device)
print(f"TinyResNet — params: {param_count(resnet):,}")


In [ ]:
hist_resnet = train_model(resnet, (train_loader, test_loader), label="TinyResNet")
plot_history(hist_resnet, "TinyResNet")


## 7. ConvNeXt block — modernised CNN (2022)

Liu et al. 2022 modernised ResNet one change at a time: **patchify stem, depthwise 7×7 conv, inverted bottleneck (expand × 4 then shrink), LayerNorm, GELU, fewer norms/activations per block**. The result matches Swin Transformer at every scale.


In [ ]:
class ConvNeXtBlock(nn.Module):
    """DWConv 7×7 → LN → Linear (×4) → GELU → Linear (÷4) + residual."""
    def __init__(self, dim, expansion=4):
        super().__init__()
        hidden = dim * expansion
        self.dwconv  = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm    = nn.LayerNorm(dim)
        self.pwconv1 = nn.Linear(dim, hidden)
        self.act     = nn.GELU()
        self.pwconv2 = nn.Linear(hidden, dim)

    def forward(self, x):
        r = x
        x = self.dwconv(x)                  # (B, C, H, W)
        x = x.permute(0, 2, 3, 1)           # → (B, H, W, C) for LN over C
        x = self.norm(x)
        x = self.pwconv1(x); x = self.act(x); x = self.pwconv2(x)
        x = x.permute(0, 3, 1, 2)           # back to (B, C, H, W)
        return x + r


class Downsample(nn.Module):
    """LayerNorm (channels-last) + strided 2×2 conv."""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.norm = nn.LayerNorm(in_dim)
        self.conv = nn.Conv2d(in_dim, out_dim, kernel_size=2, stride=2)
    def forward(self, x):
        x = x.permute(0, 2, 3, 1); x = self.norm(x); x = x.permute(0, 3, 1, 2)
        return self.conv(x)


class SimpleConvNeXt(nn.Module):
    def __init__(self, img_size=IMG_SIZE, in_ch=3, num_classes=NUM_CLASSES,
                 dims=(48, 96, 192), depths=(2, 2, 2), stem_patch=4):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, dims[0], kernel_size=stem_patch, stride=stem_patch),
            nn.LayerNorm([dims[0], img_size // stem_patch, img_size // stem_patch]),
        )
        self.stages, self.downsamples = nn.ModuleList(), nn.ModuleList()
        for i, (dim, depth) in enumerate(zip(dims, depths)):
            self.stages.append(nn.Sequential(*[ConvNeXtBlock(dim) for _ in range(depth)]))
            if i < len(dims) - 1:
                self.downsamples.append(Downsample(dim, dims[i + 1]))
        self.head_norm = nn.LayerNorm(dims[-1])
        self.head      = nn.Linear(dims[-1], num_classes)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward_features(self, x, return_maps=False):
        x = self.stem(x)
        maps = []
        for i, stage in enumerate(self.stages):
            x = stage(x)
            if return_maps: maps.append(x)
            if i < len(self.downsamples):
                x = self.downsamples[i](x)
        pooled = x.mean([-2, -1])
        pooled = self.head_norm(pooled)
        return (pooled, maps) if return_maps else pooled

    def forward(self, x):
        return self.head(self.forward_features(x))

convnext = SimpleConvNeXt().to(device)
print(f"SimpleConvNeXt — params: {param_count(convnext):,}")


In [ ]:
hist_cx = train_model(convnext, (train_loader, test_loader), label="SimpleConvNeXt")
plot_history(hist_cx, "SimpleConvNeXt")


## 8. Architecture comparison — params, FLOPs, accuracy

All five models trained with the **same loop, same data, same number of epochs**. The only difference is what's between input and head.


In [ ]:
models   = [("MLP", mlp,     hist_mlp),
            ("LeNet",     lenet,    hist_lenet),
            ("VGGTiny",   vgg,      hist_vgg),
            ("TinyResNet",resnet,   hist_resnet),
            ("ConvNeXt",  convnext, hist_cx)]

summary = []
for name, m, h in models:
    p     = param_count(m)
    flops = model_flops(m)
    summary.append({
        "model": name,
        "params": p,
        "params_M": p / 1e6,
        "GFLOPs":   flops / 1e9,
        "best_test_acc":  max(h["test_acc"]),
        "final_test_acc": h["test_acc"][-1],
    })

# Pretty-print a comparison table
print(f"{'Model':<12} {'Params':>10} {'GFLOPs':>10} {'Best test':>10} {'Final test':>11}")
print("-" * 56)
for r in summary:
    print(f"{r['model']:<12} {r['params_M']:>9.2f}M {r['GFLOPs']:>10.3f} "
          f"{r['best_test_acc']*100:>9.2f}% {r['final_test_acc']*100:>10.2f}%")


In [ ]:
# Side-by-side comparison plot.
colors = {"MLP":"#F96167","LeNet":"#7C3AED","VGGTiny":"#0891B2","TinyResNet":"#F59E0B","ConvNeXt":"#10B981"}

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy curves
for name, m, h in models:
    ep = np.arange(1, len(h["test_acc"]) + 1)
    ax[0].plot(ep, h["train_acc"], "--", color=colors[name], alpha=0.4)
    ax[0].plot(ep, h["test_acc"],  "-",  color=colors[name], lw=2.2,
               label=f"{name} (best {max(h['test_acc'])*100:.1f}%)")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("accuracy"); ax[0].set_ylim(0, 1.02)
ax[0].set_title("Train (dashed) vs Test (solid) accuracy"); ax[0].legend(loc="lower right", fontsize=9)
ax[0].grid(True, linestyle="--", linewidth=0.5, alpha=0.5)

# Params vs accuracy scatter
for r in summary:
    ax[1].scatter(r["params_M"], r["best_test_acc"]*100, s=140, color=colors[r["model"]], zorder=3)
    ax[1].annotate(r["model"], (r["params_M"], r["best_test_acc"]*100),
                   textcoords="offset points", xytext=(8, 4), fontsize=10)
ax[1].set_xscale("log"); ax[1].set_xlabel("Parameters (M)  ·  log scale")
ax[1].set_ylabel("Best test accuracy (%)"); ax[1].set_ylim(0, 105)
ax[1].set_title("Accuracy per parameter"); ax[1].grid(True, linestyle="--", linewidth=0.5, alpha=0.5)

plt.tight_layout(); plt.show()


## 9. How do we look inside a trained network?

We progress in three steps, mirroring the slides:

1. **MLP** — the simplest case. Reshape the first-layer weights back to image shape (we already did this in §3).
2. **CNN feature maps** — for deeper networks, look at the *activations* a forward pass produces, captured with a forward hook.
3. **Different aggregation strategies** — a feature map has `C` channels; pick *one* channel, the *mean*, the *max*, or the top-3 PCA components mapped to RGB.


### 9.0 Pick a sample input + see what every model predicts


In [ ]:
# Pick one input from the test set
img, true_label = test_ds[random.randint(0, len(test_ds) - 1)]
img_batch = img.unsqueeze(0).to(device)
plt.figure(figsize=(3.5, 3.5)); plt.imshow(denorm(img))
plt.title(f"Input: {CLASS_NAMES[true_label]}"); plt.axis("off"); plt.show()


In [ ]:
# Predictions from every trained model on this single input.
with torch.no_grad():
    for name, m, _ in models:
        m.eval()
        logits = m(img_batch)
        probs  = logits.softmax(-1)[0]
        topk   = probs.argsort(descending=True)[:3]
        bullet = "  ".join(f"{CLASS_NAMES[c]} ({probs[c]*100:.1f}%)" for c in topk)
        marker = "✔" if topk[0].item() == true_label else "✘"
        print(f"{marker} {name:<12}  →  {bullet}")


### 9.1 MLP — first-layer weights as image templates

We already did this in §3. The MLP can only show *what each hidden unit globally looks for*: blurry whole-image templates.


In [ ]:
# (Same visualisation as §3 — kept here so this section is self-contained.)
W = mlp.net[0].weight.detach().cpu().numpy()
fig, axes = plt.subplots(1, 10, figsize=(16, 2))
for i, ax in enumerate(axes):
    t = W[i].reshape(3, IMG_SIZE, IMG_SIZE).transpose(1, 2, 0)
    t = (t - t.min()) / (t.max() - t.min() + 1e-9)
    ax.imshow(t); ax.axis("off"); ax.set_title(f"h{i}", fontsize=9)
plt.suptitle("MLP first-layer weights — every hidden unit is a whole-image template")
plt.tight_layout(); plt.show()


### 9.2 CNN — forward hooks for intermediate activations

PyTorch lets us register a function that runs on every forward pass through a module. We use it to capture each stage's output without modifying the model.


In [ ]:
def collect_feature_maps(model, x, modules: dict):
    """Run a forward pass through `model` and return {name: feature_map} for the listed modules."""
    captured, handles = {}, []
    def make_hook(name):
        def hook(_module, _inp, out): captured[name] = out.detach().cpu()
        return hook
    for name, mod in modules.items():
        handles.append(mod.register_forward_hook(make_hook(name)))
    model.eval()
    with torch.no_grad(): _ = model(x)
    for h in handles: h.remove()
    return captured

# Pick one input from the test set
img, true_label = test_ds[random.randint(0, len(test_ds) - 1)]
img_batch = img.unsqueeze(0).to(device)
print(f"Input class: {CLASS_NAMES[true_label]}")


### 9.3 Strategy A — pick one channel

Quickest sanity check. We grab the activation tensor `(B, C, H, W)` and plot **a single channel** at each stage. Useful for spotting that the network is doing *something*, but C-channel choice is arbitrary.


In [ ]:
# Capture stages from the trained ConvNeXt
cx_modules = {f"stage{i+1}": stage for i, stage in enumerate(convnext.stages)}
fmaps = collect_feature_maps(convnext, img_batch, cx_modules)

fig, axes = plt.subplots(1, len(fmaps) + 1, figsize=(4 * (len(fmaps) + 1), 4))
axes[0].imshow(denorm(img)); axes[0].set_title("input"); axes[0].axis("off")
for i, (name, fm) in enumerate(fmaps.items()):
    axes[i+1].imshow(fm[0, 0].numpy(), cmap="viridis")
    axes[i+1].set_title(f"{name}  channel 0\nshape {tuple(fm.shape[1:])}")
    axes[i+1].axis("off")
plt.suptitle("ConvNeXt feature maps — single-channel view")
plt.tight_layout(); plt.show()


### 9.4 Strategy B — aggregate over channels

Comparing **mean**, **max**, and **PCA top-3 → RGB** for the *same* feature map.

- **Mean over C** — overall activation strength at each spatial location.
- **Max over C** — where the most-active filter fires (peakier).
- **PCA top-3** — project the C-dim feature vector at every spatial position into 3 directions of maximum variance, then display as an RGB image. Reveals the structure of the whole feature space.


In [ ]:
from sklearn.decomposition import PCA

def visualise_aggregations(fm: torch.Tensor, title: str):
    """fm: (1, C, H, W) tensor. Plot four aggregation strategies side by side."""
    C, H, W = fm.shape[1], fm.shape[2], fm.shape[3]
    arr = fm[0].numpy()                                      # (C, H, W)

    one  = arr[0]
    mean = arr.mean(0)
    mx   = arr.max(0)
    feat = arr.reshape(C, -1).T                              # (H*W, C)
    pca3 = PCA(n_components=3).fit_transform(feat).reshape(H, W, 3)
    pca3 = (pca3 - pca3.min()) / (pca3.max() - pca3.min() + 1e-9)

    fig, axes = plt.subplots(1, 4, figsize=(13, 3.3))
    for ax, im, name in zip(axes, [one, mean, mx, pca3],
                            ["channel 0", "mean over C", "max over C", "PCA top-3 → RGB"]):
        ax.imshow(im, cmap=None if im.ndim == 3 else "viridis")
        ax.set_title(name); ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout(); plt.show()

# Apply to the deepest ConvNeXt stage
visualise_aggregations(fmaps["stage3"], f"ConvNeXt stage 3  ·  {tuple(fmaps['stage3'].shape[1:])}")


### 9.5 Compare across architectures

Same input, same aggregation strategy (mean over C), one row per model. The progression should be visible: VGG and ResNet activations are spatially localised; ConvNeXt is the cleanest.


In [ ]:
# Pick the deepest stage for each model
probe_modules = {
    "VGGTiny":        vgg.features[2],     # last VGGBlock
    "TinyResNet":     resnet.stage3[-1],   # last residual block
    "SimpleConvNeXt": convnext.stages[-1], # last ConvNeXt stage
}

rows = []
for name, mod in probe_modules.items():
    parent = {"VGGTiny": vgg, "TinyResNet": resnet, "SimpleConvNeXt": convnext}[name]
    fm = collect_feature_maps(parent, img_batch, {"out": mod})["out"]
    rows.append((name, fm))

fig, axes = plt.subplots(len(rows), 4, figsize=(13, 3.4 * len(rows)))
for r, (name, fm) in enumerate(rows):
    arr = fm[0].numpy()
    C = arr.shape[0]
    feat = arr.reshape(C, -1).T
    pca3 = PCA(n_components=3).fit_transform(feat).reshape(arr.shape[1], arr.shape[2], 3)
    pca3 = (pca3 - pca3.min()) / (pca3.max() - pca3.min() + 1e-9)
    panels = [arr[0], arr.mean(0), arr.max(0), pca3]
    titles = [f"{name}: channel 0", "mean over C", "max over C", "PCA → RGB"]
    for ax, im, t in zip(axes[r], panels, titles):
        ax.imshow(im, cmap=None if im.ndim == 3 else "viridis")
        ax.set_title(t, fontsize=10); ax.axis("off")
fig.suptitle("Feature maps at the deepest stage — same input across architectures", y=1.02)
plt.tight_layout(); plt.show()


### 9.6 Confusion matrix on the test set

A different lens — instead of looking at activations, look at *errors*: which classes the best model confuses with which.


In [ ]:
# Confusion matrix on the test set for the best-performing model.
from sklearn.metrics import confusion_matrix
best = max(models, key=lambda kv: max(kv[2]["test_acc"]))
name, m, _ = best

all_preds, all_labels = [], []
m.eval()
with torch.no_grad():
    for x, y in test_loader:
        all_preds.append(m(x.to(device)).argmax(1).cpu()); all_labels.append(y)
preds  = torch.cat(all_preds).numpy()
labels = torch.cat(all_labels).numpy()
cm = confusion_matrix(labels, preds)

fig, ax = plt.subplots(figsize=(6.5, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
ax.set_yticklabels(CLASS_NAMES)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if cm[i, j]:
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=9)
ax.set_title(f"Confusion matrix — {name}")
plt.tight_layout(); plt.show()


## ✏️ Exercises

1. **MLP capacity scan.** Vary the MLP `hidden` argument (e.g. `(128,)`, `(512, 256)`, `(2048, 1024)`). Plot final test accuracy against parameter count — at what point does adding parameters stop helping?

2. **Ablate one ConvNeXt ingredient.** Pick *one* of the six modernisation steps (patchify stem · 7×7 depthwise · LayerNorm · GELU · inverted bottleneck) and re-train ConvNeXt with that change reverted to its pre-2020 ResNet-style equivalent. How much does it cost on Fruits-360?

3. **Transfer learning with a pretrained ConvNeXt-Tiny.** Use `torchvision.models.convnext_tiny(weights="DEFAULT")`, freeze the backbone, swap the classifier head to 10 classes, fine-tune for 3 epochs. Compare wall-clock time and accuracy to your from-scratch SimpleConvNeXt.

4. *(Bonus)* **Residual ablation.** Remove the `+ self.shortcut(x)` term from `BasicBlock` and re-train `TinyResNet`. Watch optimisation start failing at depth.


In [ ]:
# Your code here
